[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordADH2026PythonBootcamp/blob/main/solutions/09_L8_bridge_to_medvision.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util, subprocess, sys

if importlib.util.find_spec("adh_python") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/BradSegal/OxfordADH2026PythonBootcamp.git"],
        check=True,
    )
print("Course helpers ready.")

# Level 8: From a patient record to a model prompt

In the MedVision lab you will be presenting an AI model with a chest X-ray
and text-based question. This text is called a **prompt**. A good prompt
primes the model with some clinical context, indicates what to focus on, and
asks a specific question. Similar to a good imaging request form.

This level uses everything you've learnt so far to construct that prompt
given a patient record. You then get to read the code the lab uses which is
plain Python that you can now write!

**By the end of this level you can:**
- decide which parts of a record a model needs, and leave out the rest;
- write functions that turn a patient dict into a short summary and a
  prompt; and
- read a library's source code and answer questions about what it does.

In [ ]:
import inspect

from adh_python.checks import level8 as check
from adh_python.data import sample_patient
from adh_python.imaging import vlm

## Send only what is needed

A prompt is text that leaves your notebook. For Colab this goes to a model
running on Google's computers, for a commercial chatbot it goes to a
different company. You have shared anything you put in the prompt.

UK data protection law refers to this as **data minimisation**: use only the
personal data you need to perform a given task. The Caldicott Principles
similarly state that you should only use the minimum necessary amount of
confidential information for health and care data. Our model only needs the
age, the problem the patient is presenting with, and perhaps a few key
observations to interpret a chest X-ray. It is true that sex can be relevant
for certain queries, however, for Jordan, we do not have that information in
the record and so use the age and diagnosis. It is never necessary to
include their name or hospital number.

Here is the record you will be working with.

In [ ]:
patient = sample_patient()
for key in patient:
    print(key, "->", patient[key])

## Problem 8.1: Predict

A colleague wrote this first attempt at a summary for the model:

```python
def naive_summary(patient):
    return f"{patient['name']} ({patient['hospital_number']}), {patient['age']}, {patient['diagnosis']}"
```

**Before running anything**, make a concrete prediction about what
`naive_summary(patient)` will return for Jordan. Store this as text in
`prediction`.

In [ ]:
prediction = "Jordan Nguyen (RTH-102938), 54, Community-acquired pneumonia"
check.predict_summary(prediction)

In [ ]:
def naive_summary(patient):
    return f"{patient['name']} ({patient['hospital_number']}), {patient['age']}, {patient['diagnosis']}"


print(naive_summary(patient))

The summary works but the first two parts identify Jordan and do not help
anyone read a CXR. They can be removed without losing anything.

## Problem 8.2: Modify

`fields_to_send` is a list of keys from the record that we would include in
the prompt. Modify this list to only include what a model would need to read
a chest X-ray, namely, the age and the diagnosis. You can include
`"observations"` if you feel that it is useful.

In [ ]:
fields_to_send = ["age", "diagnosis"]
for key in fields_to_send:
    print(key, "->", patient[key])
check.minimal_fields(fields_to_send)

Minimal for privacy is not the same as neutral for reading. Sending the
diagnosis protects Jordan's identity but tells the model what to expect and
can steer what it reports. The MedVision lab tests exactly that.

Removing the name and hospital number makes the prompt **de-identified** but
not the same as anonymous. If the diagnosis is rare, if the date is exact,
or if it is in a small town, you may still be able to narrow it down to one
person. UK GDPR still considers data that could be linked to someone if
additional information is included (**pseudonymised** data) as personal
data. Only if it is **anonymous** and reasonably unlikely to be
re-identified does it fall outside of the law.

Images carry identifiers too. A hospital X-ray file (DICOM format) stores
the name, hospital number, and date of birth in the header and many films
have text burned into the picture itself. Stripping the written prompt is
not sufficient if the image still names the patient. Never place real
patient records or images into a tool your organisation has not approved.

## Problem 8.3: Make, a summary for the model

Write a function `summarise_patient(patient)` that returns a short single
line of text with the following information:

- the age and the diagnosis; and
- the **latest** respiratory rate, oxygen saturation (SpO2) and
  temperature.

Put each number straight after its label, so that a reader can tell which
number is which. It must not contain the name or hospital number, and it
must be under 300 characters. For Jordan it might return:

`54-year-old, Community-acquired pneumonia. Latest observations: respiratory rate 17/min, SpO2 96%, temperature 36.8 °C.`

The observations are stored as a dict of lists. You can try
`print(patient["observations"].keys())` to see the exact keys. The
respiratory rate, for example, is stored under `"respiration_rate"`. The
lists are ordered from oldest to latest, so the latest value would be the
last element. For example, one piece of your f-string might be:

`f"SpO2 {patient['observations']['spo2'][-1]}%"`

The checker also tries a second patient so ensure you use the record rather
than typing in Jordan's numbers.

In [ ]:
def summarise_patient(patient):
    observations = patient["observations"]
    return (
        f"{patient['age']}-year-old, {patient['diagnosis']}. "
        f"Latest observations: "
        f"respiratory rate {observations['respiration_rate'][-1]}/min, "
        f"SpO2 {observations['spo2'][-1]}%, "
        f"temperature {observations['temperature'][-1]} °C."
    )


print(summarise_patient(patient))
check.summarise_patient(summarise_patient)

## Problem 8.4: Make, the prompt

Write `build_imaging_prompt(summary, focus, question)`. It returns a prompt
of exactly four lines, with each part after its label:

```
Clinical details: <summary>
Focus: <focus>
Question: <question>
Describe only what you can see on the image, and say if you are unsure.
```

`\n` inside a string starts a new line. Copy the last line exactly; it asks
the model not to guess.

A prompt with no question is a waste of a model run so if `question` is
empty (`""`), raise a `ValueError` as you did in Level 6.

In [ ]:
def build_imaging_prompt(summary, focus, question):
    if question == "":
        raise ValueError("the prompt needs a question")
    return (
        f"Clinical details: {summary}\n"
        f"Focus: {focus}\n"
        f"Question: {question}\n"
        "Describe only what you can see on the image, and say if you are unsure."
    )


summary = summarise_patient(patient)
prompt = build_imaging_prompt(
    summary, "the lungs", "Is there consolidation, and if so, where?"
)
print(prompt)
check.build_imaging_prompt(build_imaging_prompt)

## Reading library code

The MedVision lab has a module called `vlm`. This stands for vision language
model, i.e., a model that can read both an image and text simultaneously.
This is included within the course package and is a short file with
functions similar to what you wrote above.

You can read any Python library's code. `inspect.getsource` returns the
source of a module/function as text that can be printed. Before printing it,
read the four questions of Problem 8.5 and the below notes and then proceed
to read the file with the questions in mind.

## Problem 8.5: Investigate the library

1. `env_variable`: the name of the environment variable that chooses a
   different model, as text.
2. `no_gpu_result`: The value that `load_model()` returns if there is no
   GPU.
3. `prompt_function`: text name of the function that can build a prompt
   given a summary, focus, and question.
4. `max_tokens`: the default maximum length of a reply, in tokens, as a
   number. A **token** is the unit a language model reads and writes:
   roughly three-quarters of a word.

A few things in the file are new.

- **Type hints** like `name: str` or `-> dict | None` indicate what kind of
  value a parameter takes or a function returns (`|` means "or"). Python
  does not enforce these; they are for the reader.
- **`import torch` within a function.** The heavy libraries needed to run
  the model are only loaded if a function requires them and so importing
  `vlm` is fast.
- **`os.environ.get(...)`.** An **environment variable** is a named setting
  stored outside your code that any program can read. Similar to a
  ward-level configuration rather than a single chart note, setting this to
  a different model name makes the `load_model` function pull that model
  instead without requiring any code modifications.
- **`None`**. Python's "nothing to see here" value. A function returns this
  on purpose to indicate that there isn't a result.

Do not worry about every line. Read the docstrings (the text in triple
quotes) and the names first, and scroll through the output.

In [ ]:
print(inspect.getsource(vlm))

You can also print just a single function:

In [ ]:
print(inspect.getsource(vlm.load_model))

Now answer the four questions.

In [ ]:
env_variable = "ADH_VLM_MODEL"
no_gpu_result = None
prompt_function = "build_prompt"
max_tokens = 400
check.reading_vlm(env_variable, no_gpu_result, prompt_function, max_tokens)

#### Why None and not an error?
`load_model` returns `None` rather than stopping with an error, so a
notebook without a GPU still runs: later cells see `None` and show replies
that were recorded earlier instead. It does the same if the download or
loading fails, after saying what went wrong.

## Problem 8.6: Fix the bug

This function uses the package's `vlm.build_prompt` to make a request for
any patient. It runs without an error, but the prompt it builds is wrong.
Print it, find the mistake and fix it.

Python matches arguments to parameters on a **positional basis**, i.e., the
first argument goes to the first parameter, etc. You can also name them, as
here: `vlm.build_prompt(summary, focus="the lungs", question="...")`. This
makes it impossible to get the order wrong.

In [ ]:
def imaging_request(patient, focus, question):
    summary = f"{patient['age']}-year-old, {patient['diagnosis']}"
    return vlm.build_prompt(summary, focus=focus, question=question)


print(imaging_request(patient, "the lungs", "Is there consolidation?"))
check.imaging_request(imaging_request)

`vlm.build_prompt` expects the parameters in the order of `patient_summary`,
`focus`, and then `question`. The function call here has them the wrong way
around as it passes the question as the second parameter and the focus as
the third. Python just maps these by position, so the model is asked to
focus on the question 'Is there consolidation?' and ask the question 'the
lungs'. This is all text so it does not raise an error; it is just
incorrect. A solution to this is to explicitly name the arguments in the
function call, i.e., `focus=focus, question=question`, as this cannot be
misordered.

## Your function and the package's do the same job

Compare the prompt your function builds with the one the package builds
from the same summary. `vlm.FOCUS` and `vlm.QUESTION` are the focus and
question the lab uses for its first read.

In [ ]:
summary = summarise_patient(patient)
mine = build_imaging_prompt(summary, vlm.FOCUS, vlm.QUESTION)
package = vlm.build_prompt(summary, vlm.FOCUS, vlm.QUESTION)
print(package)
print()
print("My prompt is the same as the package's:", mine == package)

If that prints `True`, your function is doing the same thing as the package.
`vlm.build_prompt` is the same as yours and `vlm.describe` adds the model
call which you don't need to follow line-by-line. The code that makes the
lab work isn't magic.

## Before you move on

- A prompt is data that leaves your control. Provide the minimal amount of
  information possible. No names, no hospital numbers, no information that
  isn't relevant to the task at hand.
- Small functions that do a single job (summarise, then build the prompt)
  are easier to verify and reuse than one long function.
- Arguments are matched by position unless named. Naming them prevents a
  whole class of silent mistakes.
- You can read any library's code with `inspect.getsource`. Read the
  docstrings and names first.

**Next:** the MedVision lab, where prompts built this way, with the
package's `build_prompt`, go to a real model.